# MLP do Zero para Reconhecimento de Sílabas — Versão Experimental

## 1. Introdução e organização dos dados

**Objetivo.** Entender *como a MLP aprende* e *como o erro se comporta* quando os hiperparâmetros mudam. Maximizar a acurácia **não** é o objetivo.

**Tarefa.** A rede classifica somente seis sílabas — `DI, REI, TA` (de DIREITA) e `ES, QUER, DA` (de ESQUERDA). As palavras são reconhecidas depois, por um verificador de sequência simples (`if/elif/else`) que lê as previsões da MLP; não existe uma segunda rede.

**Regras.** Apenas NumPy (+ `scipy.io.wavfile` para ler WAV). Inicialização, ReLU, sigmoide, softmax, feedforward, entropia cruzada, backpropagation, Adam, L2 e Early Stopping são implementados à mão. `IPython.display` serve somente para exibir os gráficos SVG.

**Dados.** 423 sílabas WAV (16 kHz) já segmentadas, divididas **por gravação**: 318 treino / 63 validação / 42 teste. A pasta `dados/` precisa estar no diretório de execução (local ou Colab). O notebook histórico `Reconhecimento_Voz_MLP_Do_Zero.ipynb` não é alterado.

In [1]:
# ==============================================================================
# CONFIGURAÇÃO DE AMBIENTE: Detecção Google Colab / Execução Local
# ==============================================================================
import os, sys, subprocess

# INSIRA AQUI A URL PÚBLICA DO SEU REPOSITÓRIO GITHUB ANTES DE EXECUTAR NO COLAB:
REPO_URL = "https://github.com/SEU_USUARIO/SEU_REPOSITORIO.git"

em_colab = "google.colab" in sys.modules

if em_colab:
    print("Ambiente detectado: Google Colab.")
    if not os.path.exists("dados/silabas.csv"):
        nome_pasta = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
        if not os.path.exists(nome_pasta):
            if "SEU_USUARIO" in REPO_URL or not REPO_URL.startswith("https://github.com/") or REPO_URL.endswith("SEU_REPOSITORIO.git"):
                print("AVISO: Configure a variável REPO_URL com a URL pública do seu repositório GitHub para clonar os dados automaticamente.")
            else:
                print(f"Clonando repositório público via HTTPS: {REPO_URL} ...")
                subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        if os.path.exists(nome_pasta):
            os.chdir(nome_pasta)
            print(f"Diretório alterado para a raiz do projeto: {os.getcwd()}")
    else:
        print(f"Diretório do projeto já configurado: {os.getcwd()}")
else:
    print(f"Ambiente detectado: Execução Local ({sys.platform}).")
    while not os.path.exists("dados/silabas.csv") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

if os.path.exists("dados/silabas.csv") and os.path.exists("dados/particoes.csv"):
    print("✔ Diretório do projeto configurado com sucesso!")
    print(f"  Raiz: {os.getcwd()}")
    print("  Arquivos 'dados/silabas.csv' e 'dados/particoes.csv' encontrados.")
else:
    print(f"✖ ATENÇÃO: Pasta 'dados/' não foi localizada a partir de {os.getcwd()}.")


Ambiente detectado: Execução Local (linux).
✔ Diretório do projeto configurado com sucesso!
  Raiz: /home/soaresgaab/Documentos/Mestrado UFMA/Disciplinas/Redes Neurais/Pipeline
  Arquivos 'dados/silabas.csv' e 'dados/particoes.csv' encontrados.


In [2]:
# Configuração, leitura dos metadados e auditoria das partições
import os, sys, csv
import numpy as np
from scipy.io import wavfile
from IPython.display import display, SVG, HTML   # somente para exibir gráficos (não é ML)

CAMINHO_DADOS = "dados"
CAMINHO_SILABAS_CSV = os.path.join(CAMINHO_DADOS, "silabas.csv")
CAMINHO_PARTICOES_CSV = os.path.join(CAMINHO_DADOS, "particoes.csv")
assert os.path.exists(CAMINHO_SILABAS_CSV) and os.path.exists(CAMINHO_PARTICOES_CSV), \
    "Pasta 'dados/' não encontrada no diretório atual."

CLASSES = ["DI", "REI", "TA", "ES", "QUER", "DA"]
CLASSE_ID = {c: i for i, c in enumerate(CLASSES)}
N_CLASSES = len(CLASSES)
PARTICOES = ["treinamento", "validacao", "teste"]

with open(CAMINHO_SILABAS_CSV, encoding="utf-8") as f:
    registros = list(csv.DictReader(f))
with open(CAMINHO_PARTICOES_CSV, encoding="utf-8") as f:
    particao_da_gravacao = {r["arquivo"]: r["conjunto"] for r in csv.DictReader(f)}

# Cada sílaba herda a partição da sua gravação: nenhuma gravação é dividida entre conjuntos.
for r in registros:
    assert particao_da_gravacao[r["pronuncia_origem"]] == r["conjunto"], r["arquivo_silaba"]

contagem = {c: {p: 0 for p in PARTICOES} for c in CLASSES}
for r in registros:
    contagem[r["classe"]][r["conjunto"]] += 1
totais = {p: sum(contagem[c][p] for c in CLASSES) for p in PARTICOES}
n_gravacoes = {p: sum(1 for v in particao_da_gravacao.values() if v == p) for p in PARTICOES}

print(f"{'Classe':<8}{'Treino':>8}{'Valid.':>8}{'Teste':>8}")
for c in CLASSES:
    print(f"{c:<8}{contagem[c]['treinamento']:>8}{contagem[c]['validacao']:>8}{contagem[c]['teste']:>8}")
print(f"{'Total':<8}{totais['treinamento']:>8}{totais['validacao']:>8}{totais['teste']:>8}   (= {sum(totais.values())} sílabas)")
print(f"Gravações por partição: {n_gravacoes}")
assert (totais["treinamento"], totais["validacao"], totais["teste"]) == (318, 63, 42)
assert sum(totais.values()) == 423

Classe    Treino  Valid.   Teste
DI            51      10       7
REI           51      10       7
TA            51      10       7
ES            55      11       7
QUER          55      11       7
DA            55      11       7
Total        318      63      42   (= 423 sílabas)
Gravações por partição: {'treinamento': 106, 'validacao': 21, 'teste': 14}


## 2. Extração de características FFT

Reutilizamos a extração do projeto: FFT do sinal inteiro da sílaba → magnitude unilateral → **média em 16 bandas de 500 Hz** (0–8 kHz) → vetor de 16 características. O pré-processamento **não muda** neste notebook, para que as diferenças observadas venham dos hiperparâmetros.

A padronização z-score usa média e desvio padrão calculados **somente no treinamento**. Em seguida verificamos dimensões e a correspondência arquivo ↔ característica ↔ rótulo.

In [3]:
# FFT (16 bandas) + padronização z-score + verificações de correspondência
def extrair_caracteristicas_fft(caminho_arquivo, n_bandas=16):
    taxa, audio = wavfile.read(caminho_arquivo)
    sinal = audio.astype(np.float64) / 32768.0 if audio.dtype == np.int16 else audio.astype(np.float64)
    n = len(sinal)
    if n == 0:
        return np.zeros(n_bandas)
    espectro = np.fft.fft(sinal)
    n_meio = n // 2 + 1
    mag = np.abs(espectro[:n_meio]) / float(n)          # espectro unilateral normalizado
    if n % 2 == 0:
        mag[1:-1] *= 2.0
    else:
        mag[1:] *= 2.0
    freq = np.linspace(0.0, taxa / 2.0, n_meio)
    largura = (taxa / 2.0) / n_bandas                    # 500 Hz por banda
    carac = np.zeros(n_bandas)
    for b in range(n_bandas):
        lo, hi = b * largura, (b + 1) * largura
        idx = np.where((freq >= lo) & ((freq <= hi) if b == n_bandas - 1 else (freq < hi)))[0]
        carac[b] = np.mean(mag[idx]) if len(idx) else 0.0
    return carac

def caminho_silaba(r):
    return os.path.join(CAMINHO_DADOS, "silabas", r["classe"], r["arquivo_silaba"])

dados = {p: {"X": [], "y": [], "meta": []} for p in PARTICOES}
for r in registros:
    taxa, audio = wavfile.read(caminho_silaba(r))
    assert taxa == 16000 and len(audio) == int(r["amostras"]), r["arquivo_silaba"]   # áudio ↔ metadados
    assert r["arquivo_silaba"].endswith(f"_{r['classe']}.wav")                         # nome ↔ classe
    d = dados[r["conjunto"]]
    d["X"].append(extrair_caracteristicas_fft(caminho_silaba(r)))
    d["y"].append(CLASSE_ID[r["classe"]])
    d["meta"].append(r)

X_tr, y_tr = np.array(dados["treinamento"]["X"]), np.array(dados["treinamento"]["y"])
X_va, y_va = np.array(dados["validacao"]["X"]),   np.array(dados["validacao"]["y"])
X_te, y_te = np.array(dados["teste"]["X"]),       np.array(dados["teste"]["y"])
meta_te = dados["teste"]["meta"]

# Padronização manual: estatísticas SOMENTE do treinamento
media_tr = X_tr.mean(axis=0, keepdims=True)
desvio_tr = X_tr.std(axis=0, keepdims=True)
desvio_tr[desvio_tr < 1e-12] = 1.0
Xp_tr, Xp_va, Xp_te = [(X - media_tr) / desvio_tr for X in (X_tr, X_va, X_te)]

# Verificações
assert Xp_tr.shape == (318, 16) and Xp_va.shape == (63, 16) and Xp_te.shape == (42, 16)
assert all(np.isfinite(A).all() for A in (Xp_tr, Xp_va, Xp_te))
assert np.allclose(Xp_tr.mean(axis=0), 0, atol=1e-9) and np.allclose(Xp_tr.std(axis=0), 1, atol=1e-9)
r0 = meta_te[0]                                               # correspondência arquivo → característica → rótulo
assert np.allclose(extrair_caracteristicas_fft(caminho_silaba(r0)), X_te[0])
assert CLASSES[y_te[0]] == r0["classe"] == os.path.basename(os.path.dirname(caminho_silaba(r0)))
print(f"X_tr {Xp_tr.shape} | X_va {Xp_va.shape} | X_te {Xp_te.shape}  (padronizados com média/desvio do treino)")
print(f"Exemplo de correspondência: {r0['arquivo_silaba']} → classe {CLASSES[y_te[0]]} → partição {r0['conjunto']}")

X_tr (318, 16) | X_va (63, 16) | X_te (42, 16)  (padronizados com média/desvio do treino)
Exemplo de correspondência: dir_04_DI.wav → classe DI → partição teste


## 3. Implementação manual da MLP

Uma única implementação, parametrizada por **camadas ocultas, ativação, taxa de aprendizado, épocas, L2, paciência e semente**; todos os experimentos a reutilizam.

- **Inicialização:** He para ReLU (`σ² = 2/n_in`), Xavier/Glorot para sigmoide (`σ² = 2/(n_in+n_out)`); *bias* nulos.
- **Saída:** softmax estável + entropia cruzada. Gradiente da saída: `(P − Y)/N`.
- **Adam manual** com correção de viés; **L2** acoplado ao gradiente apenas nos pesos (`+ λW`, nunca nos *biases*).
- **Early Stopping** monitora a perda de validação (entropia cruzada, sem o termo L2, para comparar configurações de forma justa) e restaura os melhores pesos.
- **Medições:** perda e acurácia (treino e validação) são registradas **depois** de cada atualização dos pesos. O treinamento usa o lote completo, então 1 época = 1 atualização.

*Mudança em relação ao notebook histórico (documentada):* lá a perda de treino era medida **antes** da atualização e a de validação **depois**; aqui ambas são medidas depois.

In [4]:
# MLP manual: ativações, inicialização, feedforward, perda, backpropagation, Adam e treinamento
def relu(Z):            return np.maximum(0.0, Z)
def d_relu(Z):          return (Z > 0.0).astype(np.float64)
def sigmoide(Z):
    saida = np.empty_like(Z)
    pos = Z >= 0                                   # forma estável para Z muito negativo/positivo
    saida[pos] = 1.0 / (1.0 + np.exp(-Z[pos]))
    ez = np.exp(Z[~pos])
    saida[~pos] = ez / (1.0 + ez)
    return saida
def d_sigmoide(Z):
    s = sigmoide(Z)
    return s * (1.0 - s)
def softmax(Z):
    e = np.exp(Z - np.max(Z, axis=1, keepdims=True))   # subtrai o máximo: estabilidade numérica
    return e / np.sum(e, axis=1, keepdims=True)

ATIVACOES = {"relu": (relu, d_relu), "sigmoide": (sigmoide, d_sigmoide)}

def inicializar_parametros(camadas, semente=42, ativacao="relu", init=None):
    """init: 'he' (padrão p/ ReLU) ou 'xavier' (padrão p/ sigmoide)."""
    init = init or ("he" if ativacao == "relu" else "xavier")
    rng = np.random.default_rng(semente)
    p = {}
    for l in range(1, len(camadas)):
        n_in, n_out = camadas[l - 1], camadas[l]
        desvio = np.sqrt(2.0 / n_in) if init == "he" else np.sqrt(2.0 / (n_in + n_out))
        p[f"W{l}"] = rng.normal(0.0, desvio, size=(n_in, n_out))
        p[f"b{l}"] = np.zeros((1, n_out))
    return p

def n_camadas(p):
    return len(p) // 2

def feedforward(X, p, ativacao="relu"):
    f = ATIVACOES[ativacao][0]
    L = n_camadas(p)
    cache = {"A0": X}
    A = X
    for l in range(1, L + 1):
        Z = A @ p[f"W{l}"] + p[f"b{l}"]
        A = softmax(Z) if l == L else f(Z)
        cache[f"Z{l}"], cache[f"A{l}"] = Z, A
    return A, cache

def entropia_cruzada(P, y, eps=1e-15):
    Ps = np.clip(P, eps, 1.0 - eps)
    return -np.mean(np.log(Ps[np.arange(len(y)), y]))

def penalidade_l2(p, lam):
    return 0.5 * lam * sum(np.sum(p[f"W{l}"] ** 2) for l in range(1, n_camadas(p) + 1))

def backpropagation(X, y, p, cache, ativacao="relu", lam=0.0):
    d_f = ATIVACOES[ativacao][1]
    L, N = n_camadas(p), len(y)
    Y = np.zeros_like(cache[f"A{L}"])
    Y[np.arange(N), y] = 1.0
    dZ = (cache[f"A{L}"] - Y) / N                       # softmax + entropia cruzada
    g = {}
    for l in range(L, 0, -1):
        g[f"W{l}"] = cache[f"A{l-1}"].T @ dZ + lam * p[f"W{l}"]   # + gradiente da penalidade L2
        g[f"b{l}"] = dZ.sum(axis=0, keepdims=True)
        if l > 1:
            dZ = (dZ @ p[f"W{l}"].T) * d_f(cache[f"Z{l-1}"])      # regra da cadeia
    return g

def atualizar_adam(p, g, m, v, t, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    novo = {}
    for k in p:
        m[k] = beta1 * m[k] + (1 - beta1) * g[k]
        v[k] = beta2 * v[k] + (1 - beta2) * g[k] ** 2
        m_hat = m[k] / (1 - beta1 ** t)
        v_hat = v[k] / (1 - beta2 ** t)
        novo[k] = p[k] - lr * m_hat / (np.sqrt(v_hat) + eps)
    return novo

def acuracia(P, y):
    return 100.0 * np.mean(np.argmax(P, axis=1) == y)

def treinar_mlp(X_tr, y_tr, X_va, y_va, ocultas=(32, 16), ativacao="relu", lr=0.01,
                max_epocas=300, lam=0.0, paciencia=20, semente=42, init=None, min_delta=1e-5):
    camadas = [X_tr.shape[1], *ocultas, N_CLASSES]
    p = inicializar_parametros(camadas, semente, ativacao, init)
    m = {k: np.zeros_like(a) for k, a in p.items()}
    v = {k: np.zeros_like(a) for k, a in p.items()}
    hist = {"perda_treino": [], "perda_val": [], "acc_treino": [], "acc_val": []}
    melhor, melhor_epoca, melhores, sem_melhora, parou = np.inf, 0, None, 0, False
    for ep in range(1, max_epocas + 1):
        _, cache = feedforward(X_tr, p, ativacao)
        p = atualizar_adam(p, backpropagation(X_tr, y_tr, p, cache, ativacao, lam), m, v, ep, lr)
        P_tr, _ = feedforward(X_tr, p, ativacao)        # medições APÓS a atualização
        P_va, _ = feedforward(X_va, p, ativacao)
        hist["perda_treino"].append(entropia_cruzada(P_tr, y_tr))
        hist["perda_val"].append(entropia_cruzada(P_va, y_va))
        hist["acc_treino"].append(acuracia(P_tr, y_tr))
        hist["acc_val"].append(acuracia(P_va, y_va))
        if melhor - hist["perda_val"][-1] > min_delta:
            melhor, melhor_epoca, sem_melhora = hist["perda_val"][-1], ep, 0
            melhores = {k: a.copy() for k, a in p.items()}
        else:
            sem_melhora += 1
            if sem_melhora >= paciencia:
                parou = True
                break
    i = melhor_epoca - 1
    return {"melhores_params": melhores, "hist": hist, "melhor_epoca": melhor_epoca,
            "epoca_final": ep, "parou_early_stopping": parou, "camadas": camadas,
            "ativacao": ativacao, "lam": lam, "lr": lr, "semente": semente,
            "n_params": sum(a.size for a in melhores.values()),
            "val_final": {"perda": hist["perda_val"][i], "acc": hist["acc_val"][i],
                          "perda_treino": hist["perda_treino"][i], "acc_treino": hist["acc_treino"][i]}}

print("✔ MLP manual definida (ReLU/sigmoide, softmax, entropia cruzada, backprop, Adam, L2, Early Stopping).")

✔ MLP manual definida (ReLU/sigmoide, softmax, entropia cruzada, backprop, Adam, L2, Early Stopping).


In [5]:
# Teste numérico dos gradientes (diferenças finitas centrais) — ReLU e sigmoide, com e sem L2
def verificar_gradientes(ativacao, lam, ocultas=(32, 16), n_amostras=6, epsilon=1e-6):
    camadas = [16, *ocultas, N_CLASSES]
    p = inicializar_parametros(camadas, semente=7, ativacao=ativacao)
    X, y = Xp_tr[:n_amostras], y_tr[:n_amostras]
    _, cache = feedforward(X, p, ativacao)
    g_ana = backpropagation(X, y, p, cache, ativacao, lam)

    def J(par):                                     # perda total = entropia cruzada + L2
        P, _ = feedforward(X, par, ativacao)
        return entropia_cruzada(P, y) + penalidade_l2(par, lam)

    erros = {}
    for k in p:
        g_num = np.zeros_like(p[k])
        for idx in np.ndindex(*p[k].shape):
            orig = p[k][idx]
            p[k][idx] = orig + epsilon; j_mais = J(p)
            p[k][idx] = orig - epsilon; j_menos = J(p)
            p[k][idx] = orig
            g_num[idx] = (j_mais - j_menos) / (2 * epsilon)
        erros[k] = np.linalg.norm(g_ana[k] - g_num) / (np.linalg.norm(g_ana[k]) + np.linalg.norm(g_num) + 1e-12)
    return max(erros.values())

print(f"{'Ativação':<10}{'λ (L2)':>8}{'Erro relativo máximo':>24}")
for ativ, lam in [("relu", 0.0), ("relu", 1e-2), ("sigmoide", 0.0), ("sigmoide", 1e-2)]:
    e = verificar_gradientes(ativ, lam)
    print(f"{ativ:<10}{lam:>8g}{e:>24.2e}   {'OK' if e < 1e-5 else 'FALHOU'}")
    assert e < 1e-5, f"Gradiente inconsistente ({ativ}, λ={lam}): {e}"
print("✔ Backpropagation confere com o gradiente numérico (tolerância 1e-5).")

Ativação    λ (L2)    Erro relativo máximo
relu             0                9.09e-10   OK
relu          0.01                9.09e-10   OK
sigmoide         0                1.26e-08   OK
sigmoide      0.01                1.42e-08   OK
✔ Backpropagation confere com o gradiente numérico (tolerância 1e-5).


## 4. Treinamento inicial da rede

Configuração de referência: **16 → 32 → 16 → 6**, ReLU (He), Adam manual, `lr = 0,010`, até 300 épocas, Early Stopping com paciência 20, semente 42, sem L2.

In [6]:
CONFIG_REF = dict(ocultas=(32, 16), ativacao="relu", lr=0.010, max_epocas=300, lam=0.0, paciencia=20, semente=42)

res_ref = treinar_mlp(Xp_tr, y_tr, Xp_va, y_va, **CONFIG_REF)
vf = res_ref["val_final"]
print(f"Arquitetura {'→'.join(map(str, res_ref['camadas']))} | {res_ref['n_params']} parâmetros")
print(f"Melhor época (menor perda de validação): {res_ref['melhor_epoca']}")
print(f"Treinamento terminou na época {res_ref['epoca_final']} "
      f"({'Early Stopping' if res_ref['parou_early_stopping'] else 'limite de épocas'})")
print(f"Na melhor época → treino: perda {vf['perda_treino']:.4f}, acc {vf['acc_treino']:.2f}% | "
      f"validação: perda {vf['perda']:.4f}, acc {vf['acc']:.2f}%")

Arquitetura 16→32→16→6 | 1174 parâmetros
Melhor época (menor perda de validação): 103
Treinamento terminou na época 123 (Early Stopping)
Na melhor época → treino: perda 0.1517, acc 95.60% | validação: perda 0.0863, acc 95.24%


## 5. Análise das curvas de erro e aprendizado

Os gráficos são SVGs gerados com NumPy/Python (sem Matplotlib). A leitura esperada:

- **Aprendeu?** a perda de treino e de validação diminuem nas primeiras épocas.
- **Estagnou?** as curvas se achatam.
- **Sobreajuste?** a perda de treino continua caindo enquanto a de validação sobe, ou a distância entre as duas cresce.

In [7]:
# Utilitários de gráficos SVG (NumPy/Python puro): linhas, barras e matriz de confusão
import math
PALETA = ["#2563eb", "#dc2626", "#16a34a", "#d97706", "#7c3aed", "#0891b2", "#be185d"]

def _esc(t):
    return str(t).replace("&", "&amp;").replace("<", "&lt;")

def _ticks(vmin, vmax, n=5):
    span = (vmax - vmin) if vmax > vmin else 1.0
    bruto = span / n
    mag = 10 ** math.floor(math.log10(bruto))
    passo = next(f * mag for f in (1, 2, 2.5, 5, 10) if bruto <= f * mag * 1.0001)
    t, saida = math.ceil(vmin / passo - 1e-9) * passo, []
    while t <= vmax + 1e-9:
        saida.append(round(t, 10)); t += passo
    return saida

def _moldura(W, H, titulo):
    return (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" style="width:100%;height:auto;'
            f'background:#fff;border:1px solid #e2e8f0;border-radius:6px;font-family:sans-serif">'
            f'<text x="{W/2}" y="22" text-anchor="middle" font-size="13" font-weight="bold" fill="#1e293b">{_esc(titulo)}</text>')

def svg_linhas(series, titulo, xlabel, ylabel, ymin=0.0, ymax=None, xmax=None, vlinhas=(), W=640, H=340):
    """series: lista de dict(nome, x, y, cor, tracejado). vlinhas: lista de (x, rotulo, cor)."""
    ml, mr, mt, mb = 62, 205, 38, 50
    pw, ph = W - ml - mr, H - mt - mb
    xmax = xmax or max(max(s["x"]) for s in series)
    ymax = ymax or max(max(s["y"]) for s in series) * 1.05
    X = lambda x: ml + (x / xmax) * pw
    Y = lambda y: mt + ph - ((y - ymin) / (ymax - ymin)) * ph
    o = [_moldura(W, H, titulo)]
    for t in _ticks(ymin, ymax):
        o.append(f'<line x1="{ml}" y1="{Y(t):.1f}" x2="{ml+pw}" y2="{Y(t):.1f}" stroke="#e2e8f0"/>'
                 f'<text x="{ml-6}" y="{Y(t)+4:.1f}" text-anchor="end" font-size="10" fill="#475569">{t:.3g}</text>')
    for t in _ticks(0, xmax, 6):
        o.append(f'<line x1="{X(t):.1f}" y1="{mt+ph}" x2="{X(t):.1f}" y2="{mt+ph+4}" stroke="#64748b"/>'
                 f'<text x="{X(t):.1f}" y="{mt+ph+16}" text-anchor="middle" font-size="10" fill="#475569">{t:g}</text>')
    o.append(f'<rect x="{ml}" y="{mt}" width="{pw}" height="{ph}" fill="none" stroke="#64748b"/>')
    o.append(f'<text x="{ml+pw/2}" y="{H-8}" text-anchor="middle" font-size="11" fill="#334155">{_esc(xlabel)}</text>')
    o.append(f'<text transform="translate(14,{mt+ph/2}) rotate(-90)" text-anchor="middle" font-size="11" fill="#334155">{_esc(ylabel)}</text>')
    for xv, rot, cor in vlinhas:
        o.append(f'<line x1="{X(xv):.1f}" y1="{mt}" x2="{X(xv):.1f}" y2="{mt+ph}" stroke="{cor}" stroke-dasharray="5,4"/>'
                 f'<text x="{X(xv)+(-4 if X(xv) > ml + pw * 0.6 else 4):.1f}" y="{mt+12}" text-anchor="{"end" if X(xv) > ml + pw * 0.6 else "start"}" font-size="10" fill="{cor}">{_esc(rot)}</text>')
    for i, s in enumerate(series):
        cor = s.get("cor", PALETA[i % len(PALETA)])
        dash = ' stroke-dasharray="6,3"' if s.get("tracejado") else ""
        pts = " ".join(f"{X(a):.1f},{Y(min(max(b, ymin), ymax)):.1f}" for a, b in zip(s["x"], s["y"]))
        o.append(f'<polyline points="{pts}" fill="none" stroke="{cor}" stroke-width="1.8"{dash}/>')
        ly = mt + 10 + i * 17
        o.append(f'<line x1="{ml+pw+12}" y1="{ly}" x2="{ml+pw+34}" y2="{ly}" stroke="{cor}" stroke-width="2"{dash}/>'
                 f'<text x="{ml+pw+39}" y="{ly+4}" font-size="10" fill="#334155">{_esc(s["nome"])}</text>')
    o.append("</svg>")
    return "".join(o)

def svg_barras(categorias, series, titulo, ylabel, ymax=None, W=640, H=340, decimais=3):
    """series: lista de dict(nome, valores, cor, erros=None, pontos=None). Rótulos de categoria: use '|' para quebrar linha."""
    ml, mr, mt, mb = 62, 150 if len(series) > 1 else 20, 38, 62
    pw, ph = W - ml - mr, H - mt - mb
    topo = max(max(v + (s["erros"][j] if s.get("erros") else 0) for j, v in enumerate(s["valores"])) for s in series)
    topo = max([topo] + [q for s in series if s.get("pontos") for ps in s["pontos"] for q in ps])
    ymax = ymax or topo * 1.18
    Y = lambda y: mt + ph - (y / ymax) * ph
    nc, ns = len(categorias), len(series)
    gw = pw / nc
    bw = gw * 0.72 / ns
    o = [_moldura(W, H, titulo)]
    for t in _ticks(0, ymax):
        o.append(f'<line x1="{ml}" y1="{Y(t):.1f}" x2="{ml+pw}" y2="{Y(t):.1f}" stroke="#e2e8f0"/>'
                 f'<text x="{ml-6}" y="{Y(t)+4:.1f}" text-anchor="end" font-size="10" fill="#475569">{t:.3g}</text>')
    o.append(f'<rect x="{ml}" y="{mt}" width="{pw}" height="{ph}" fill="none" stroke="#64748b"/>')
    o.append(f'<text transform="translate(14,{mt+ph/2}) rotate(-90)" text-anchor="middle" font-size="11" fill="#334155">{_esc(ylabel)}</text>')
    for j, cat in enumerate(categorias):
        cx = ml + gw * (j + 0.5)
        for k, linha in enumerate(str(cat).split("|")):
            o.append(f'<text x="{cx:.1f}" y="{mt+ph+14+12*k}" text-anchor="middle" font-size="10" fill="#334155">{_esc(linha)}</text>')
        for i, s in enumerate(series):
            cor = s["cores_barras"][j] if s.get("cores_barras") else s.get("cor", PALETA[i % len(PALETA)])
            x0 = cx - (bw * ns) / 2 + i * bw
            v = s["valores"][j]
            o.append(f'<rect x="{x0:.1f}" y="{Y(v):.1f}" width="{bw*0.92:.1f}" height="{max(0.0, mt+ph-Y(v)):.1f}" fill="{cor}" fill-opacity="0.75"/>')
            xc = x0 + bw * 0.46
            if s.get("erros"):
                e = s["erros"][j]
                o.append(f'<line x1="{xc:.1f}" y1="{Y(v-e):.1f}" x2="{xc:.1f}" y2="{Y(v+e):.1f}" stroke="#0f172a" stroke-width="1.5"/>'
                         f'<line x1="{xc-4:.1f}" y1="{Y(v+e):.1f}" x2="{xc+4:.1f}" y2="{Y(v+e):.1f}" stroke="#0f172a" stroke-width="1.5"/>')
            if s.get("pontos"):
                for q in s["pontos"][j]:
                    o.append(f'<circle cx="{xc:.1f}" cy="{Y(q):.1f}" r="2.8" fill="#0f172a" fill-opacity="0.85"/>')
            if s.get("erros"):
                o.append(f'<text x="{xc:.1f}" y="{mt+ph-6:.1f}" text-anchor="middle" font-size="10" font-weight="bold" fill="#0f172a">{v:.{decimais}f}</text>')
            else:
                o.append(f'<text x="{xc:.1f}" y="{Y(v)-5:.1f}" text-anchor="middle" font-size="10" fill="#1e293b">{v:.{decimais}f}</text>')
    if ns > 1:
        for i, s in enumerate(series):
            ly = mt + 10 + i * 17
            o.append(f'<rect x="{ml+pw+12}" y="{ly-6}" width="14" height="10" fill="{s.get("cor", PALETA[i % len(PALETA)])}" fill-opacity="0.75"/>'
                     f'<text x="{ml+pw+32}" y="{ly+3}" font-size="10" fill="#334155">{_esc(s["nome"])}</text>')
    o.append("</svg>")
    return "".join(o)

def svg_confusao(M, classes, titulo, W=520, H=480):
    ml, mt, c = 90, 78, 62
    o = [_moldura(W, H, titulo)]
    o.append(f'<text x="{ml+c*len(classes)/2}" y="{mt-34}" text-anchor="middle" font-size="11" fill="#334155">Classe prevista</text>')
    o.append(f'<text transform="translate(16,{mt+c*len(classes)/2}) rotate(-90)" text-anchor="middle" font-size="11" fill="#334155">Classe real</text>')
    for i, ci in enumerate(classes):
        o.append(f'<text x="{ml-8}" y="{mt+i*c+c/2+4}" text-anchor="end" font-size="11" font-weight="bold" fill="#334155">{ci}</text>')
        o.append(f'<text x="{ml+i*c+c/2}" y="{mt-10}" text-anchor="middle" font-size="11" font-weight="bold" fill="#334155">{ci}</text>')
        soma = max(1, int(M[i].sum()))
        for j in range(len(classes)):
            v = int(M[i, j])
            if i == j:
                fundo, txt = f"rgba(37,99,235,{0.12 + 0.8 * v / soma:.2f})", ("#fff" if v / soma > 0.6 else "#1e3a8a")
            else:
                fundo, txt = (f"rgba(220,38,38,{min(0.85, 0.25 + 0.3 * v):.2f})" if v else "#fff"), ("#7f1d1d" if v else "#94a3b8")
            o.append(f'<rect x="{ml+j*c}" y="{mt+i*c}" width="{c}" height="{c}" fill="{fundo}" stroke="#cbd5e1"/>'
                     f'<text x="{ml+j*c+c/2}" y="{mt+i*c+c/2+5}" text-anchor="middle" font-size="15" font-weight="bold" fill="{txt}">{v}</text>')
    o.append("</svg>")
    return "".join(o)

def mostrar(svg, largura_max=640):
    display(HTML(f'<div style="max-width:{largura_max}px">{svg}</div>'))

def mostrar_lado_a_lado(*svgs):
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:8px">'
                 + "".join(f'<div style="flex:1 1 440px;min-width:0">{s}</div>' for s in svgs) + "</div>"))
print("✔ Utilitários de gráficos SVG definidos.")

✔ Utilitários de gráficos SVG definidos.


In [8]:
# Curvas da configuração de referência + diagnóstico objetivo
h, ep_opt, ep_fim = res_ref["hist"], res_ref["melhor_epoca"], res_ref["epoca_final"]
ep = list(range(1, ep_fim + 1))
vl = [(ep_opt, f"melhor época {ep_opt}", "#16a34a")]
g_perda = svg_linhas([dict(nome="Treino", x=ep, y=h["perda_treino"], cor="#2563eb"),
                      dict(nome="Validação", x=ep, y=h["perda_val"], cor="#dc2626")],
                     "Perda (entropia cruzada) — referência", "Época", "Perda", vlinhas=vl, W=720, H=360)
g_acc = svg_linhas([dict(nome="Treino", x=ep, y=h["acc_treino"], cor="#2563eb"),
                    dict(nome="Validação", x=ep, y=h["acc_val"], cor="#dc2626")],
                   "Acurácia — referência", "Época", "Acurácia (%)", ymin=0, ymax=105, vlinhas=vl, W=720, H=360)
mostrar_lado_a_lado(g_perda, g_acc)

i = ep_opt - 1
queda = 100 * (1 - h["perda_treino"][i] / h["perda_treino"][0])
print(f"Perda de treino: {h['perda_treino'][0]:.3f} (época 1) → {h['perda_treino'][i]:.3f} (melhor época), queda de {queda:.1f}%")
print(f"Perda de validação: mínimo {h['perda_val'][i]:.4f} na época {ep_opt}; ao fim (época {ep_fim}): {h['perda_val'][-1]:.4f}")
print(f"Distância validação − treino na melhor época: {h['perda_val'][i] - h['perda_treino'][i]:+.4f}; "
      f"na última época: {h['perda_val'][-1] - h['perda_treino'][-1]:+.4f}")
print(f"Variação da perda de treino nas últimas 20 épocas: {h['perda_treino'][-1] - h['perda_treino'][-20]:+.4f}")

Perda de treino: 1.797 (época 1) → 0.152 (melhor época), queda de 91.6%
Perda de validação: mínimo 0.0863 na época 103; ao fim (época 123): 0.0889
Distância validação − treino na melhor época: -0.0654; na última época: -0.0290
Variação da perda de treino nas últimas 20 épocas: -0.0326


## 6. Experimentos de hiperparâmetros

Cada grupo muda **uma variável por vez** e mantém o resto da configuração de referência (16→32→16→6, ReLU, `lr = 0,010`, sem L2, 300 épocas, paciência 20). Cada configuração é treinada com as **sementes 42, 43 e 44**; as tabelas mostram média ± desvio padrão amostral (`n−1`) entre as sementes. Todas as decisões usam **apenas treino e validação**.

| Grupo | Variável | Valores | Por quê |
|---|---|---|---|
| A | Taxa de aprendizado | 0,005 · 0,010 · 0,020 | passo menor/maior que o de referência |
| B | Ativação × inicialização | ReLU+He · ReLU+Xavier · Sigmoide+Xavier · Sigmoide+He | separar o efeito da função do efeito da escala inicial |
| C | Camadas ocultas | (16,8) · (32,16) · (48,24) | menor/maior capacidade (462 · 1174 · 2142 parâmetros) |
| D | Regularização L2 (λ) | 0 · 1e-4 · 1e-3 · 1e-2 | a penalidade `½λ‖W‖²` passa de desprezível a comparável à perda de entropia cruzada (~0,1–0,3) |

No gráfico de cada grupo: à esquerda, curvas da semente 42 (treino tracejado, validação contínua); à direita, perda de validação na melhor época (barra = média, barra de erro = desvio, pontos = cada semente).

In [9]:
# Infraestrutura dos experimentos (reutiliza treinar_mlp) + Grupo A: taxa de aprendizado
SEMENTES = (42, 43, 44)
_CACHE = {}

def rodar(ocultas=(32, 16), ativacao="relu", lr=0.010, lam=0.0, semente=42, init=None):
    init = init or ("he" if ativacao == "relu" else "xavier")
    chave = (tuple(ocultas), ativacao, lr, lam, semente, init)
    if chave not in _CACHE:
        _CACHE[chave] = treinar_mlp(Xp_tr, y_tr, Xp_va, y_va, ocultas=tuple(ocultas), ativacao=ativacao,
                                    lr=lr, max_epocas=300, lam=lam, paciencia=20, semente=semente, init=init)
        _CACHE[chave]["init"] = init
    return _CACHE[chave]

def rodar_grupo(configs):
    return {nome: [rodar(semente=s, **kw) for s in SEMENTES] for nome, kw in configs.items()}

def resumir(runs):
    arr = lambda f: np.array([f(r) for r in runs], dtype=float)
    perda, acc = arr(lambda r: r["val_final"]["perda"]), arr(lambda r: r["val_final"]["acc"])
    p_tr = arr(lambda r: r["val_final"]["perda_treino"])
    pos = arr(lambda r: r["hist"]["perda_val"][-1] - r["val_final"]["perda"])   # quanto a perda de val subiu após a melhor época
    return dict(n_params=runs[0]["n_params"], perdas=perda, perda_m=perda.mean(), perda_s=perda.std(ddof=1),
                acc_m=acc.mean(), acc_s=acc.std(ddof=1), perda_tr_m=p_tr.mean(), gap_m=(perda - p_tr).mean(),
                ep_melhor_m=arr(lambda r: r["melhor_epoca"]).mean(), ep_fim_m=arr(lambda r: r["epoca_final"]).mean(),
                n_es=int(sum(r["parou_early_stopping"] for r in runs)), pos_m=pos.mean())

def tabela_grupo(resultados):
    print(f"{'Configuração':<24}{'Params':>7}  {'Perda val':>15}  {'Acc val (%)':>14}  {'Perda trein.':>12}  {'Gap v−t':>8}  {'Ép.melhor':>9}  {'Ép.fim':>6}  ES")
    for nome, runs in resultados.items():
        s = resumir(runs)
        print(f"{nome:<24}{s['n_params']:>7}  {s['perda_m']:>7.4f}±{s['perda_s']:.4f}  {s['acc_m']:>6.2f}±{s['acc_s']:<5.2f}  "
              f"{s['perda_tr_m']:>12.4f}  {s['gap_m']:>+8.4f}  {s['ep_melhor_m']:>9.1f}  {s['ep_fim_m']:>6.1f}  {s['n_es']}/3")

def painel_grupo(titulo, resultados):
    series, nomes = [], list(resultados)
    for i, (nome, runs) in enumerate(resultados.items()):
        h, cor = runs[0]["hist"], PALETA[i % len(PALETA)]            # semente 42
        ep = list(range(1, len(h["perda_val"]) + 1))
        series.append(dict(nome=f"{nome} (val)", x=ep, y=h["perda_val"], cor=cor))
        series.append(dict(nome=f"{nome} (treino)", x=ep, y=h["perda_treino"], cor=cor, tracejado=True))
    topo = max(max(s["y"][:5]) for s in series)
    g1 = svg_linhas(series, f"{titulo}: perda por época (semente 42)", "Época", "Perda", ymax=topo * 1.05, W=720, H=360)
    res = [resumir(r) for r in resultados.values()]
    g2 = svg_barras([n.replace(" + ", "|").replace(" (", "|(") for n in nomes],
                    [dict(nome="Perda val", valores=[s["perda_m"] for s in res], erros=[s["perda_s"] for s in res],
                          pontos=[list(s["perdas"]) for s in res], cor="#2563eb")],
                    f"{titulo}: perda de validação (melhor época)", "Perda de validação", W=720, H=360)
    mostrar_lado_a_lado(g1, g2)

CONFIGS_A = {"lr = 0.005": dict(lr=0.005), "lr = 0.010 (ref.)": dict(lr=0.010), "lr = 0.020": dict(lr=0.020)}
res_A = rodar_grupo(CONFIGS_A)
tabela_grupo(res_A)
painel_grupo("A — Taxa de aprendizado", res_A)

Configuração             Params        Perda val     Acc val (%)  Perda trein.   Gap v−t  Ép.melhor  Ép.fim  ES
lr = 0.005                 1174   0.1551±0.0302   93.12±0.92         0.1783   -0.0232      152.7   172.7  3/3
lr = 0.010 (ref.)          1174   0.1342±0.0417   93.65±1.59         0.1843   -0.0501       87.0   107.0  3/3
lr = 0.020                 1174   0.1256±0.0098   96.30±1.83         0.1894   -0.0638       55.0    75.0  3/3


In [10]:
# Grupo B: função de ativação × inicialização
CONFIGS_B = {"ReLU + He (ref.)":    dict(ativacao="relu", init="he"),
             "ReLU + Xavier":       dict(ativacao="relu", init="xavier"),
             "Sigmoide + Xavier":   dict(ativacao="sigmoide", init="xavier"),
             "Sigmoide + He":       dict(ativacao="sigmoide", init="he")}
res_B = rodar_grupo(CONFIGS_B)
tabela_grupo(res_B)
painel_grupo("B — Ativação e inicialização", res_B)

Configuração             Params        Perda val     Acc val (%)  Perda trein.   Gap v−t  Ép.melhor  Ép.fim  ES
ReLU + He (ref.)           1174   0.1342±0.0417   93.65±1.59         0.1843   -0.0501       87.0   107.0  3/3
ReLU + Xavier              1174   0.0933±0.0214   96.30±0.92         0.1371   -0.0438      107.3   127.3  3/3
Sigmoide + Xavier          1174   0.1083±0.0180   96.30±1.83         0.2010   -0.0927      294.3   300.0  0/3
Sigmoide + He              1174   0.1060±0.0199   96.83±1.59         0.1974   -0.0914      287.7   294.3  1/3


In [11]:
# Grupo C: arquitetura (camadas ocultas)
CONFIGS_C = {"16-8":          dict(ocultas=(16, 8)),
             "32-16 (ref.)":  dict(ocultas=(32, 16)),
             "48-24":         dict(ocultas=(48, 24))}
res_C = rodar_grupo(CONFIGS_C)
tabela_grupo(res_C)
painel_grupo("C — Arquitetura", res_C)

Configuração             Params        Perda val     Acc val (%)  Perda trein.   Gap v−t  Ép.melhor  Ép.fim  ES
16-8                        462   0.0876±0.0277   96.83±1.59         0.1570   -0.0694      178.3   198.3  3/3
32-16 (ref.)               1174   0.1342±0.0417   93.65±1.59         0.1843   -0.0501       87.0   107.0  3/3
48-24                      2142   0.1215±0.0204   95.77±2.42         0.1735   -0.0519       75.7    95.7  3/3


In [12]:
# Grupo D: regularização L2
CONFIGS_D = {"λ = 0 (ref.)": dict(lam=0.0), "λ = 1e-4": dict(lam=1e-4),
             "λ = 1e-3": dict(lam=1e-3),    "λ = 1e-2": dict(lam=1e-2)}
res_D = rodar_grupo(CONFIGS_D)
tabela_grupo(res_D)
painel_grupo("D — Regularização L2", res_D)

Configuração             Params        Perda val     Acc val (%)  Perda trein.   Gap v−t  Ép.melhor  Ép.fim  ES
λ = 0 (ref.)               1174   0.1342±0.0417   93.65±1.59         0.1843   -0.0501       87.0   107.0  3/3
λ = 1e-4                   1174   0.1332±0.0387   93.65±1.59         0.1683   -0.0351       94.3   114.3  3/3
λ = 1e-3                   1174   0.1315±0.0250   94.71±0.92         0.1651   -0.0336      104.7   124.7  3/3
λ = 1e-2                   1174   0.1656±0.0051   95.77±0.92         0.2621   -0.0964      267.7   286.3  2/3


### Grupo E — combinações justificadas pelos resultados A–D

Nos grupos A–D, mudanças *isoladas* reduziram a perda de validação média em relação à referência: **rede menor (16-8)**, **inicialização Xavier** com ReLU e **taxa maior (0,020)**, esta também com menor dispersão entre sementes; um **L2 moderado (1e-3)** reduziu a dispersão, enquanto λ = 1e-2 piorou o resultado. Como esses efeitos podem se somar ou se anular, testamos apenas **quatro combinações** (sem busca exaustiva).

In [13]:
# Grupo E: combinações dos efeitos observados em A–D
CONFIGS_E = {"16-8 + Xavier":                 dict(ocultas=(16, 8), init="xavier"),
             "16-8 + Xavier + lr 0.020":      dict(ocultas=(16, 8), init="xavier", lr=0.020),
             "32-16 + Xavier + lr 0.020":     dict(ocultas=(32, 16), init="xavier", lr=0.020),
             "16-8 + Xavier + λ=1e-3":        dict(ocultas=(16, 8), init="xavier", lam=1e-3)}
res_E = rodar_grupo(CONFIGS_E)
tabela_grupo(res_E)
painel_grupo("E — Combinações", res_E)

Configuração             Params        Perda val     Acc val (%)  Perda trein.   Gap v−t  Ép.melhor  Ép.fim  ES
16-8 + Xavier               462   0.0992±0.0158   96.30±0.92         0.1568   -0.0576      177.0   197.0  3/3
16-8 + Xavier + lr 0.020    462   0.0958±0.0272   96.83±1.59         0.1688   -0.0729      103.3   123.3  3/3
32-16 + Xavier + lr 0.020   1174   0.0876±0.0088   96.83±1.59         0.1597   -0.0721       70.7    90.7  3/3
16-8 + Xavier + λ=1e-3      462   0.0840±0.0074   96.30±0.92         0.1416   -0.0577      201.7   221.7  3/3


## 7. Comparação e seleção da configuração

A seleção **não** usa a acurácia de treino nem o conjunto de teste. A regra, aplicada em código sobre todas as configurações treinadas com as três sementes, é:

1. **Convergência:** só são candidatas configurações em que o Early Stopping interrompeu o treino nas três sementes (sem bater no limite de 300 épocas).
2. **Perda de validação:** é candidata quem tem perda média de validação até **1 desvio padrão** acima da melhor média — diferenças menores que a variação entre sementes são tratadas como **empate prático**, sem afirmar superioridade.
3. **Complexidade:** entre as candidatas, escolhe-se a de **menos parâmetros** (desempate: menor perda média).

Também são mostrados, para análise, a distância validação − treino (sobreajuste), a subida da perda de validação após a melhor época e as épocas até convergir.

In [14]:
# Comparação de todas as configurações (treino e validação apenas) e seleção
def rotulo_config(k):
    oc, at, lr, lam, init = k
    l2 = f"{'ReLU' if at == 'relu' else 'Sig'} {'He' if init == 'he' else 'Xav'}"
    partes = ["-".join(map(str, oc)), l2, f"lr {lr:g}"] + ([f"λ {lam:g}"] if lam else [])
    return "|".join(partes)

unicas = {}
for (oc, at, lr, lam, sem, init), r in _CACHE.items():
    unicas.setdefault((oc, at, lr, lam, init), []).append(r)
unicas = {k: sorted(v, key=lambda r: r["semente"]) for k, v in unicas.items() if len(v) == len(SEMENTES)}
resumos = {k: resumir(v) for k, v in unicas.items()}

melhor_k = min(resumos, key=lambda k: resumos[k]["perda_m"])
limiar = resumos[melhor_k]["perda_m"] + resumos[melhor_k]["perda_s"]
candidatas = [k for k in resumos if resumos[k]["n_es"] == len(SEMENTES) and resumos[k]["perda_m"] <= limiar]
selecionada = min(candidatas, key=lambda k: (resumos[k]["n_params"], resumos[k]["perda_m"]))

ordem = sorted(resumos, key=lambda k: resumos[k]["perda_m"])
print(f"Melhor perda média de validação: {resumos[melhor_k]['perda_m']:.4f} (±{resumos[melhor_k]['perda_s']:.4f}) → limiar de empate prático = {limiar:.4f}\n")
print(f"{'Configuração':<40}{'Params':>7}  {'Perda val':>15}  {'Acc val':>6}  {'Gap v−t':>8}  {'Subida pós-melhor':>17}  {'Ép.melhor':>9}  {'ES':>3}  Status")
for k in ordem:
    s = resumos[k]
    status = "SELECIONADA" if k == selecionada else ("candidata" if k in candidatas else ("limite de épocas" if s["n_es"] < len(SEMENTES) else ""))
    print(f"{rotulo_config(k).replace('|', ' / '):<40}{s['n_params']:>7}  {s['perda_m']:>7.4f}±{s['perda_s']:.4f}  {s['acc_m']:>6.2f}  {s['gap_m']:>+8.4f}  {s['pos_m']:>17.4f}  {s['ep_melhor_m']:>9.1f}  {s['n_es']:>2}/3  {status}")

cores = ["#16a34a" if k == selecionada else ("#2563eb" if k in candidatas else "#94a3b8") for k in ordem]
nomes_barras = [rotulo_config(k) + ("|(ref.)" if k == ((32, 16), "relu", 0.01, 0.0, "he") else "") for k in ordem]
mostrar(svg_barras(nomes_barras,
    [dict(nome="Perda val", valores=[resumos[k]["perda_m"] for k in ordem], erros=[resumos[k]["perda_s"] for k in ordem],
          pontos=[list(resumos[k]["perdas"]) for k in ordem], cores_barras=cores)],
    "Perda de validação na melhor época: média ± desvio (pontos = sementes 42, 43, 44) — verde: selecionada · azul: candidata · cinza: demais",
    "Perda de validação", W=1100, H=360), 1200)

CONFIG_SEL = dict(zip(("ocultas", "ativacao", "lr", "lam", "init"), selecionada))
runs_sel = unicas[selecionada]
ref_k = ((32, 16), "relu", 0.01, 0.0, "he")
s_sel, s_ref = resumos[selecionada], resumos[ref_k]
print(f"\nSelecionada: {rotulo_config(selecionada).replace('|', ' / ')} ({s_sel['n_params']} parâmetros) — "
      f"perda val {s_sel['perda_m']:.4f}±{s_sel['perda_s']:.4f}, acc val {s_sel['acc_m']:.2f}%, "
      f"convergência em ~{s_sel['ep_melhor_m']:.0f} épocas.")
print(f"Referência (32-16, ReLU He, lr 0.01):   perda val {s_ref['perda_m']:.4f}±{s_ref['perda_s']:.4f}, acc val {s_ref['acc_m']:.2f}%, ~{s_ref['ep_melhor_m']:.0f} épocas.")
print(f"{len(candidatas)} configurações são praticamente equivalentes na perda de validação (empate dentro de 1 desvio padrão).")
SELECAO_CONGELADA = True      # a partir daqui o teste pode ser consultado; a escolha não muda

Melhor perda média de validação: 0.0840 (±0.0074) → limiar de empate prático = 0.0914

Configuração                             Params        Perda val  Acc val   Gap v−t  Subida pós-melhor  Ép.melhor   ES  Status
16-8 / ReLU Xav / lr 0.01 / λ 0.001         462   0.0840±0.0074   96.30   -0.0577             0.0074      201.7   3/3  SELECIONADA
16-8 / ReLU He / lr 0.01                    462   0.0876±0.0277   96.83   -0.0694             0.0083      178.3   3/3  candidata
32-16 / ReLU Xav / lr 0.02                 1174   0.0876±0.0088   96.83   -0.0721             0.0338       70.7   3/3  candidata
32-16 / ReLU Xav / lr 0.01                 1174   0.0933±0.0214   96.30   -0.0438             0.0099      107.3   3/3  
16-8 / ReLU Xav / lr 0.02                   462   0.0958±0.0272   96.83   -0.0729             0.0114      103.3   3/3  
16-8 / ReLU Xav / lr 0.01                   462   0.0992±0.0158   96.30   -0.0576             0.0083      177.0   3/3  
32-16 / Sig He / lr 0.01             

In [15]:
# Curvas da configuração selecionada (3 sementes)
cores_s = {42: "#2563eb", 43: "#d97706", 44: "#16a34a"}
sp, sa, vl = [], [], []
for r in runs_sel:
    h, c = r["hist"], cores_s[r["semente"]]
    ep = list(range(1, len(h["perda_val"]) + 1))
    sp += [dict(nome=f"sem. {r['semente']} val", x=ep, y=h["perda_val"], cor=c),
           dict(nome=f"sem. {r['semente']} treino", x=ep, y=h["perda_treino"], cor=c, tracejado=True)]
    sa += [dict(nome=f"sem. {r['semente']} val", x=ep, y=h["acc_val"], cor=c),
           dict(nome=f"sem. {r['semente']} treino", x=ep, y=h["acc_treino"], cor=c, tracejado=True)]
    vl.append((r["melhor_epoca"], "", c))
nome_sel = rotulo_config(selecionada).replace("|", " ")
topo = max(s["y"][4] for s in sp)
mostrar_lado_a_lado(
    svg_linhas(sp, f"Selecionada ({nome_sel}): perda", "Época", "Perda", ymax=topo * 1.05, vlinhas=vl, W=720, H=360),
    svg_linhas(sa, "Selecionada: acurácia", "Época", "Acurácia (%)", ymin=0, ymax=105, vlinhas=vl, W=720, H=360))
for r in runs_sel:
    print(f"semente {r['semente']}: melhor época {r['melhor_epoca']}, fim {r['epoca_final']}, "
          f"val perda {r['val_final']['perda']:.4f} / acc {r['val_final']['acc']:.2f}% | treino perda {r['val_final']['perda_treino']:.4f} / acc {r['val_final']['acc_treino']:.2f}%")

semente 42: melhor época 203, fim 223, val perda 0.0786 / acc 96.83% | treino perda 0.1411 / acc 94.34%
semente 43: melhor época 220, fim 240, val perda 0.0808 / acc 95.24% | treino perda 0.1223 / acc 96.54%
semente 44: melhor época 182, fim 202, val perda 0.0925 / acc 96.83% | treino perda 0.1615 / acc 95.28%


## 8. Teste das seis sílabas (avaliação retrospectiva)

Os pesos da configuração selecionada (semente 42, melhor época do Early Stopping) são **congelados**: não há nenhuma nova atualização. Avaliamos as 42 sílabas de teste.

> **Atenção:** o conjunto de teste já foi consultado nos experimentos históricos do projeto. Esta é uma avaliação **retrospectiva** — ele não participou da escolha dos hiperparâmetros *deste* notebook, mas **não** é um teste inédito.

In [16]:
# Avaliação retrospectiva no teste: pesos congelados (sem treinamento)
assert SELECAO_CONGELADA
modelo = runs_sel[0]                                    # semente 42
params_final, ativ_final = modelo["melhores_params"], modelo["ativacao"]
P_te, _ = feedforward(Xp_te, params_final, ativ_final)
prev_te = np.argmax(P_te, axis=1)
acertos = int(np.sum(prev_te == y_te))
print(f"Teste (retrospectivo): {acertos}/{len(y_te)} acertos = {100*acertos/len(y_te):.2f}% | perda {entropia_cruzada(P_te, y_te):.4f}")

M = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
for real, prev in zip(y_te, prev_te):
    M[real, prev] += 1
mostrar(svg_confusao(M, CLASSES, "Matriz de confusão — teste (42 sílabas, retrospectivo)"), 520)

print(f"\n{'Classe':<7}{'Suporte':>8}{'Acertos':>8}{'Erros':>6}{'Precisão':>10}{'Revocação':>11}")
for i, c in enumerate(CLASSES):
    tp, sup, col = M[i, i], M[i].sum(), M[:, i].sum()
    prec = f"{100*tp/col:.1f}%" if col else "—"
    print(f"{c:<7}{sup:>8}{tp:>8}{sup-tp:>6}{prec:>10}{100*tp/sup:>10.1f}%")
conf = sorted(((M[i, j], CLASSES[i], CLASSES[j]) for i in range(N_CLASSES) for j in range(N_CLASSES) if i != j and M[i, j] > 0), reverse=True)
print("\nConfusões (real → prevista):", ", ".join(f"{a}→{b} ({n}x)" for n, a, b in conf) if conf else "nenhuma")
print("\nInformativo (após a seleção; não usado para escolher): acurácia de teste da mesma configuração com outras sementes:")
for r in runs_sel[1:]:
    Pr, _ = feedforward(Xp_te, r["melhores_params"], r["ativacao"])
    print(f"  semente {r['semente']}: {100*np.mean(np.argmax(Pr, axis=1) == y_te):.2f}%")

Teste (retrospectivo): 37/42 acertos = 88.10% | perda 0.7096

Classe  Suporte Acertos Erros  Precisão  Revocação
DI            7       5     2     83.3%      71.4%
REI           7       6     1     75.0%      85.7%
TA            7       6     1     85.7%      85.7%
ES            7       7     0    100.0%     100.0%
QUER          7       6     1    100.0%      85.7%
DA            7       7     0     87.5%     100.0%

Confusões (real → prevista): DI→REI (2x), TA→DA (1x), REI→TA (1x), QUER→DI (1x)

Informativo (após a seleção; não usado para escolher): acurácia de teste da mesma configuração com outras sementes:
  semente 43: 85.71%
  semente 44: 88.10%


## 9. Reconhecimento das palavras: MLP + verificador de sequência

**Integração entre MLP e Verificador de Sequência:**
Nesta etapa, o áudio completo da palavra **não** é fornecido diretamente à MLP. O reconhecimento das palavras ocorre em conjunto com um verificador de regras:

- **Entrada da rede:** A MLP recebe separadamente as 16 características FFT de cada sílaba individualmente (não a palavra inteira).
- **Agrupamento e ordem temporal:** As previsões da rede são agrupadas pela gravação de origem (`pronuncia_origem`), e a ordem temporal das três sílabas é recuperada pelo campo `ordem` (1, 2 e 3) do arquivo de metadados (`dados/silabas.csv`).
- **Verificação da sequência:** Um verificador determinístico (`if/elif/else`) confere as sequências esperadas:
  - `DI → REI → TA` $\longrightarrow$ **DIREITA**
  - `ES → QUER → DA` $\longrightarrow$ **ESQUERDA**
  - Qualquer outra sequência $\longrightarrow$ **NÃO RECONHECIDO** (rejeição segura)

Dessa forma, a **MLP é responsável exclusivamente pelo reconhecimento acústico** de cada sílaba, enquanto o **verificador valida a coerência da sequência temporal**.

In [17]:
# Verificador de sequência (if/elif/else) aplicado às previsões da mesma MLP
def verificar_sequencia(s1, s2, s3):
    if s1 == "DI" and s2 == "REI" and s3 == "TA":
        return "DIREITA"
    elif s1 == "ES" and s2 == "QUER" and s3 == "DA":
        return "ESQUERDA"
    else:
        return "NÃO RECONHECIDO"

gravacoes = {}
for r, p in zip(meta_te, prev_te):
    g = gravacoes.setdefault(r["pronuncia_origem"], {"palavra": r["palavra"], "silabas": []})
    assert g["palavra"] == r["palavra"]
    g["silabas"].append((int(r["ordem"]), CLASSES[p]))

resultados = []
for nome in sorted(gravacoes):
    g = gravacoes[nome]
    seq = [s for _, s in sorted(g["silabas"])]                       # ordem temporal preservada
    assert len(seq) == 3 and sorted(o for o, _ in g["silabas"]) == [1, 2, 3]
    decisao = verificar_sequencia(*seq)
    status = "ACERTO" if decisao == g["palavra"] else ("REJEIÇÃO" if decisao == "NÃO RECONHECIDO" else "TROCA INCORRETA")
    resultados.append(dict(gravacao=nome, real=g["palavra"], seq=seq, decisao=decisao, status=status))

col_id   = "Palavra de origem (identificador)"
col_real = "Palavra verdadeira"
col_seq  = "Três sílabas previstas pela MLP"
col_dec  = "Decisão do verificador"
col_res  = "Resultado"

w_id, w_real, w_seq, w_dec, w_res = 36, 20, 34, 26, 16
cabecalho = f"{col_id:<{w_id}}{col_real:<{w_real}}{col_seq:<{w_seq}}{col_dec:<{w_dec}}{col_res}"
divisor = "-" * len(cabecalho)

print(cabecalho)
print(divisor)
for r in resultados:
    seq_str = " → ".join(r["seq"])
    print(f"{r['gravacao']:<{w_id}}{r['real']:<{w_real}}{seq_str:<{w_seq}}{r['decisao']:<{w_dec}}{r['status']}")
print(divisor)

contagem_pal = {p: {"ACERTO": 0, "REJEIÇÃO": 0, "TROCA INCORRETA": 0} for p in ("DIREITA", "ESQUERDA")}
for r in resultados:
    contagem_pal[r["real"]][r["status"]] += 1
tot = {k: sum(contagem_pal[p][k] for p in contagem_pal) for k in ("ACERTO", "REJEIÇÃO", "TROCA INCORRETA")}
n = len(resultados)
print(f"\nGravações: {n} | acertos {tot['ACERTO']} ({100*tot['ACERTO']/n:.1f}%) | "
      f"rejeições {tot['REJEIÇÃO']} ({100*tot['REJEIÇÃO']/n:.1f}%) | "
      f"trocas DIREITA↔ESQUERDA {tot['TROCA INCORRETA']}")

cats = ["DIREITA", "ESQUERDA", "Total"]
vals = lambda k: [contagem_pal["DIREITA"][k], contagem_pal["ESQUERDA"][k], tot[k]]
mostrar(svg_barras(cats,
    [dict(nome="Acertos", valores=vals("ACERTO"), cor="#16a34a"),
     dict(nome="Rejeições", valores=vals("REJEIÇÃO"), cor="#d97706"),
     dict(nome="Trocas incorretas", valores=vals("TROCA INCORRETA"), cor="#dc2626")],
    "Reconhecimento de palavras no teste (MLP + verificador, retrospectivo)", "Nº de gravações", decimais=0, W=640), 640)

Palavra de origem (identificador)   Palavra verdadeira  Três sílabas previstas pela MLP   Decisão do verificador    Resultado
-----------------------------------------------------------------------------------------------------------------------------
dir_04.wav                          DIREITA             DI → REI → DA                     NÃO RECONHECIDO           REJEIÇÃO
dir_09.wav                          DIREITA             DI → REI → TA                     DIREITA                   ACERTO
dir_15.wav                          DIREITA             DI → REI → TA                     DIREITA                   ACERTO
dir_29.wav                          DIREITA             REI → REI → TA                    NÃO RECONHECIDO           REJEIÇÃO
dir_32.wav                          DIREITA             DI → TA → TA                      NÃO RECONHECIDO           REJEIÇÃO
dir_36.wav                          DIREITA             REI → REI → TA                    NÃO RECONHECIDO           REJEIÇÃO
di

## 10. Conclusões experimentais

Os números exatos estão nas saídas acima; aqui ficam as leituras qualitativas, todas baseadas em treino e validação (exceto onde indicado como retrospectivo).

- **A rede aprendeu.** A perda de treino cai de forma acentuada e depois se achata; o Early Stopping interrompe o treino logo após a melhor época. Em média, em todas as configurações a perda de validação foi *menor* que a de treino e subiu pouco após a melhor época — **não há sinal claro de sobreajuste**. A validação ser "mais fácil" que o treino sugere um conjunto pequeno (63 sílabas, 21 gravações), e não necessariamente melhor generalização.
- **Taxa de aprendizado:** taxas maiores convergiram em menos épocas e 0,005 foi a mais lenta e a de maior perda média. A diferença entre 0,010 e 0,020 é menor que a variação entre sementes: é uma *tendência*, não uma superioridade comprovada.
- **Ativação:** com 300 épocas, a sigmoide ainda não tinha convergido na maioria das sementes (limite de épocas atingido), então a comparação com a ReLU é limitada. A inicialização (He × Xavier) pouco alterou a sigmoide; para a ReLU, Xavier teve perda média menor que He — não investigamos a causa.
- **Arquitetura:** a rede menor (16-8) não foi pior que as maiores; 48-24 não trouxe ganho. Com apenas 318 amostras de treino, mais parâmetros não ajudaram.
- **L2:** valores pequenos (1e-4, 1e-3) alteraram pouco a perda média e reduziram a distância validação − treino; λ = 1e-2 foi forte demais (perda de treino alta, treino longo — subajuste).
- **Seleção:** entre configurações *praticamente equivalentes* na perda de validação, foi escolhida a de menor complexidade. Com 3 sementes e 63 amostras de validação, **não afirmamos superioridade estatística** entre as candidatas; a escolha entre várias configurações na mesma validação também pode estar levemente otimista.
- **Teste (retrospectivo):** o teste já havia sido consultado em experimentos históricos, e com 42 sílabas (14 gravações) cada erro vale ~2,4 pontos percentuais. As acurácias com outras sementes variam, então diferenças pequenas em relação ao notebook histórico não são conclusivas.
- **Palavras:** o verificador exige as três sílabas corretas e na ordem, então erros silábicos viram **rejeições** (não trocas). Os erros se concentraram em sílabas de DIREITA (principalmente DI confundida com REI).
- **Limites:** um único locutor, partição por gravação e segmentação heurística; a generalização para novos locutores **não foi avaliada**.

In [18]:
# Auditoria final (sem métricas fixadas manualmente)
proibidos = ["sklearn", "torch", "tensorflow", "keras", "jax"]
assert not [m for m in proibidos if m in sys.modules], "Biblioteca de ML detectada!"
assert Xp_tr.shape == (318, 16) and Xp_va.shape == (63, 16) and Xp_te.shape == (42, 16)
assert sum(a.size for a in params_final.values()) == modelo["n_params"]
assert set(g for g in gravacoes) == {r["pronuncia_origem"] for r in meta_te}      # só gravações de teste
n_cfg, n_runs = len(unicas), len(_CACHE)
print(f"✔ Sem bibliotecas de ML (apenas NumPy/SciPy-wavfile; IPython só para exibir SVG).")
print(f"✔ Partições: {Xp_tr.shape[0]} treino / {Xp_va.shape[0]} validação / {Xp_te.shape[0]} teste.")
print(f"✔ {n_cfg} configurações × {len(SEMENTES)} sementes = {n_runs} treinamentos; seleção feita antes de consultar o teste.")
print(f"✔ Modelo final: {'→'.join(map(str, modelo['camadas']))} ({modelo['n_params']} parâmetros), "
      f"{modelo['ativacao']}, lr {modelo['lr']}, λ {modelo['lam']}, semente {modelo['semente']}, melhor época {modelo['melhor_epoca']}.")

✔ Sem bibliotecas de ML (apenas NumPy/SciPy-wavfile; IPython só para exibir SVG).
✔ Partições: 318 treino / 63 validação / 42 teste.
✔ 15 configurações × 3 sementes = 45 treinamentos; seleção feita antes de consultar o teste.
✔ Modelo final: 16→16→8→6 (462 parâmetros), relu, lr 0.01, λ 0.001, semente 42, melhor época 203.
